# Lab: Build a Simple MCP Server

In this lab you build a minimal **Model Context Protocol (MCP)** server from scratch using **FastAPI** and **Pydantic**.
The server exposes a single `/mcp` endpoint that understands two verbs:

| Verb | What happens |
|------|--------------|
| `discovery` | Returns a list of all available tools with their names, descriptions, and parameters |
| `execute` | Runs a named tool with the supplied arguments and returns the result |

**What you will build:**

```
app/
├── models.py   # Pydantic schemas for requests and responses
├── tools.py    # Tool function + Tool definition object
└── main.py     # FastAPI app wiring everything together
```

**Prerequisites:** Python 3.11+, basic FastAPI/Pydantic familiarity.

## Table of Contents

1. [Step 1 — Set Up the Environment](#step-1--set-up-the-environment)
2. [Step 2 — Define the Data Models](#step-2--define-the-data-models)
3. [Step 3 — Implement a Tool](#step-3--implement-a-tool)
4. [Step 4 — Wire Up the FastAPI Endpoint](#step-4--wire-up-the-fastapi-endpoint)
5. [Step 5 — Run and Test Manually](#step-5--run-and-test-manually)
6. [Step 6 — Write Automated Tests](#step-6--write-automated-tests)
7. [Exercises](#exercises)
8. [Solution](#solution)

## Step 1 — Set Up the Environment

Create a virtual environment and install the dependencies.

```bash
cd labs/01-build-simple-mcp-server
python -m venv .venv
source .venv/bin/activate   # Windows: .venv\Scripts\activate
pip install -r requirements.txt
```

The `requirements.txt` contains:

In [ ]:
# Print requirements.txt
with open('requirements.txt') as f:
    print(f.read())

Copy the environment template:

```bash
cp .env.example .env
```

The `.env` file holds configuration values loaded at runtime:

| Variable | Default | Purpose |
|----------|---------|--------|
| `APP_HOST` | `127.0.0.1` | Bind address |
| `APP_PORT` | `8000` | Listening port |
| `APP_ENV` | `development` | Runtime environment |

## Step 2 — Define the Data Models

`app/models.py` defines the Pydantic schemas for every request and response the server handles.

In [ ]:
# Read and display app/models.py
with open('app/models.py') as f:
    print(f.read())

**Model breakdown:**

| Class | Role |
|-------|------|
| `ToolParameter` | Describes one parameter of a tool: its `name` and `type` (e.g. `"string"`, `"number"`) |
| `Tool` | A complete tool definition: `name`, `description`, and a list of `ToolParameter` objects — this is what clients receive during `discovery` |
| `ModelContextRequest` | Incoming request body — `verb` is required; `tool_name` and `arguments` are only needed for `execute` |
| `ModelContextResponse` | Outgoing response — `tools` is populated on `discovery`; `result` is populated on `execute` |

> **Why Pydantic?** Pydantic validates and parses the JSON body automatically. If a request is missing `verb` or sends the wrong type, FastAPI returns a `422 Unprocessable Entity` before your code runs.

## Step 3 — Implement a Tool

`app/tools.py` has two responsibilities: the **function** that does the work, and the **Tool object** that describes it to clients.

In [ ]:
with open('app/tools.py') as f:
    print(f.read())

**Key concepts:**

- `get_weather(location)` is a plain Python function — no MCP-specific code inside it.
- `GET_WEATHER_TOOL` is a `Tool` instance built from the same `models.py` schemas the `/mcp` endpoint returns. The client uses this object to know what arguments to pass.
- Keeping the function and its definition in the same file makes it easy to keep them in sync.

**The separation of concerns:**

```
tools.py     ← what the tool does + what it looks like
main.py      ← how requests are routed to tools
models.py    ← the shared language between client and server
```

## Step 4 — Wire Up the FastAPI Endpoint

`app/main.py` connects everything: it creates the FastAPI app, defines the tool registry, and implements the `/mcp` route.

In [ ]:
with open('app/main.py') as f:
    print(f.read())

**How the endpoint works:**

```
POST /mcp
│
├── verb == "discovery"
│   └── return ModelContextResponse(tools=[GET_WEATHER_TOOL])
│
├── verb == "execute"
│   ├── look up tool_name in tool_registry
│   ├── call the function with arguments
│   └── return ModelContextResponse(result=...)
│
└── anything else → 400 Bad Request
```

**`tool_registry`** is a plain dict mapping tool names to functions:

```python
tool_registry = {
    "get_weather": get_weather
}
```

When you add a new tool, you add it here *and* to `ALL_TOOLS` — that is the only wiring required.

> **Why `KeyError` → 404?** A `KeyError` means the requested tool name is not in the registry. From the client's perspective this is "not found", so 404 is the correct HTTP status.

## Step 5 — Run and Test Manually

Start the server in one terminal:

```bash
uvicorn app.main:app --reload --host 127.0.0.1 --port 8000
```

### Test with curl

**Discovery:**
```bash
curl -s -X POST http://127.0.0.1:8000/mcp \
  -H "Content-Type: application/json" \
  -d '{"verb": "discovery"}' | python -m json.tool
```

**Execute `get_weather`:**
```bash
curl -s -X POST http://127.0.0.1:8000/mcp \
  -H "Content-Type: application/json" \
  -d '{"verb": "execute", "tool_name": "get_weather", "arguments": {"location": "Berlin"}}' \
  | python -m json.tool
```

**Unknown tool (expect 404):**
```bash
curl -s -X POST http://127.0.0.1:8000/mcp \
  -H "Content-Type: application/json" \
  -d '{"verb": "execute", "tool_name": "fly_to_moon"}'
```

### Test with httpx (Python)

Run the cell below while the server is running:

In [ ]:
import httpx

BASE = "http://127.0.0.1:8000"

# Discovery
r = httpx.post(f"{BASE}/mcp", json={"verb": "discovery"})
print("Discovery:", r.json())

# Execute
r = httpx.post(f"{BASE}/mcp", json={
    "verb": "execute",
    "tool_name": "get_weather",
    "arguments": {"location": "Tokyo"}
})
print("Execute:", r.json())

### Interactive API docs

FastAPI auto-generates interactive documentation — open these in a browser while the server is running:

- **Swagger UI:** http://127.0.0.1:8000/docs
- **ReDoc:** http://127.0.0.1:8000/redoc

## Step 6 — Write Automated Tests

The `tests/test_mcp.py` file uses FastAPI's `TestClient` (backed by `httpx`) — no running server needed.

In [ ]:
with open('tests/test_mcp.py') as f:
    print(f.read())

Run the full suite:

```bash
pytest tests/ -v
```

**What each test checks:**

| Test | Assertion |
|------|-----------|
| `test_discovery_returns_tools` | Response contains a `tools` list with one entry |
| `test_discovery_tool_has_parameters` | The tool has a `location` parameter |
| `test_execute_get_weather` | Result contains the location name |
| `test_execute_unknown_tool_returns_404` | Unknown tool name → 404 |
| `test_unknown_verb_returns_400` | Unknown verb → 400 |
| `test_execute_without_arguments_uses_empty_dict` | Arguments defaults to `{}` not `None` |

## Exercises

### Exercise 1 — Add an `add_numbers` tool

Add a new tool to `app/tools.py` that takes two numbers `a` and `b` and returns their sum as a string.

Requirements:
- Function signature: `def add_numbers(a: float, b: float) -> str`
- Create an `ADD_NUMBERS_TOOL` definition with two parameters of type `"number"`
- Register it in `main.py`'s `tool_registry` and include it in the discovery response

Test it:
```bash
curl -s -X POST http://127.0.0.1:8000/mcp \
  -H "Content-Type: application/json" \
  -d '{"verb": "execute", "tool_name": "add_numbers", "arguments": {"a": 3, "b": 4}}'
# Expected result: "7.0"
```

<details>
<summary>Hint</summary>

```python
def add_numbers(a: float, b: float) -> str:
    """Adds two numbers and returns the result."""
    return str(a + b)

ADD_NUMBERS_TOOL = Tool(
    name="add_numbers",
    description="Adds two numbers and returns the result as a string.",
    parameters=[
        ToolParameter(name="a", type="number"),
        ToolParameter(name="b", type="number"),
    ]
)
```
</details>

---

### Exercise 2 — Add a `ping` verb

Add a third verb `"ping"` that returns `ModelContextResponse(status="pong")`.

Requirements:
- Add a `status: Optional[str] = None` field to `ModelContextResponse` in `models.py`
- Handle `verb == "ping"` in `main.py` before the final `raise HTTPException`

Test it:
```bash
curl -s -X POST http://127.0.0.1:8000/mcp \
  -H "Content-Type: application/json" \
  -d '{"verb": "ping"}'
# Expected: {"tools": null, "result": null, "status": "pong"}
```

---

### Exercise 3 — Validate `tool_name` on execute

Currently, sending `{"verb": "execute"}` without a `tool_name` raises an unhandled `KeyError` (because `None` is not in the registry).

Fix `main.py` to return a `400` with a clear message when `verb == "execute"` but `tool_name` is missing or `None`.

<details>
<summary>Hint</summary>

```python
elif request.verb == "execute":
    if not request.tool_name:
        raise HTTPException(status_code=400, detail="'tool_name' is required for verb 'execute'.")
    ...
```
</details>

## Solution

The complete solution with all three exercises is in the `solution/` directory.

Run it on port 8001 to compare with your implementation:

In [ ]:
# Print the solution main.py
with open('solution/app/main.py') as f:
    print(f.read())

In [ ]:
# Print the solution tools.py
with open('solution/app/tools.py') as f:
    print(f.read())

Start the solution server:

```bash
uvicorn solution.app.main:app --reload --port 8001
```